In [ ]:
# Phase 10: DINOv3 ViT-L/16 features for the whole corpus (4,407 studies, 9 windows x 6 slots at 336px. Pass conditions: NOTES 2026-10-06.
import glob, os, shutil, sys, time

GIT_SHA = 'de03b2c'

SRC = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)[0]
PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')

import numpy as np
import pandas as pd
import timm
import torch

assert torch.cuda.is_available()
major, minor = torch.cuda.get_device_capability(0)
print(f'GPU: {torch.cuda.get_device_name(0)}, sm_{major}{minor}, timm {timm.__version__}, torch {torch.__version__}')
assert (major, minor) >= (7, 0), 'need T4, not P100'

from knee.dataset import PreppedSlotDataset
from knee.dicom import SLOTS
from knee.features import PooledBackbone, extract_study_features, load_feature_cache, save_feature_cache
assert any('dinov3' in n for n in timm.list_models()), 'this timm has no DINOv3'
print('src carries the Phase 10 feature modules | GIT_SHA', GIT_SHA)


In [ ]:
# The prepped corpus, mounted from the four prep9 shard kernels (as every confirm kernel does).
PREPPED_DIRS = sorted(glob.glob('/kaggle/input/**/prepped', recursive=True))
uid_to_npz = {}
for d in PREPPED_DIRS:
    for f in os.listdir(d):
        if f.endswith('.npz'):
            uid_to_npz[f[:-4]] = os.path.join(d, f)
NPZ_ROOT = '/kaggle/working/prepped_all'
os.makedirs(NPZ_ROOT, exist_ok=True)
for uid, p in uid_to_npz.items():
    dst = os.path.join(NPZ_ROOT, f'{uid}.npz')
    if not os.path.exists(dst):
        os.symlink(p, dst)
print(f'{len(PREPPED_DIRS)} shards, {len(uid_to_npz)} artifacts')
assert len(uid_to_npz) > 4000, 'shards do not cover the corpus'

slot_names = [n for n, _, _ in SLOTS]
assert len(slot_names) == 6
uids = sorted(uid_to_npz)
assert len(uids) == 4407, len(uids)
print(f'corpus: {len(uids)} studies')


In [ ]:
# The checkpoint. No internet here, so the weights come from the Kaggle Model; strict loading is the point:
# a silent partial load would leave random layers and every downstream number would be noise.
files = sorted(p for p in glob.glob('/kaggle/input/**/*', recursive=True)
               if os.path.isfile(p) and '/dinov3' in p.lower())
for p in files[:30]:
    print(f'  {p}  {os.path.getsize(p) / 2**20:.0f} MB')
weights = [p for p in files if p.endswith(('.safetensors', '.pth', '.pt', '.bin'))]
assert weights, 'no weight file found under the DINOv3 model mount'
if len(weights) > 1:
    prefer = [p for p in weights if any(k in os.path.basename(p).lower() for k in ('vitl', 'large'))]
    assert len(prefer) == 1, f'ambiguous weight files, pick one by hand: {weights}'
    weights = prefer
WEIGHTS = weights[0]
print('weights:', WEIGHTS)

if WEIGHTS.endswith('.safetensors'):
    from safetensors.torch import load_file
    state = load_file(WEIGHTS)
else:
    state = torch.load(WEIGHTS, map_location='cpu', weights_only=True)
for wrapper in ('state_dict', 'model', 'teacher'):
    if isinstance(state, dict) and wrapper in state and isinstance(state[wrapper], dict):
        state = state[wrapper]
for prefix in ('module.', 'model.', 'backbone.'):
    if state and all(k.startswith(prefix) for k in state):
        state = {k[len(prefix):]: v for k, v in state.items()}
print(f'{len(state)} tensors, first keys: {list(state)[:6]}')

from timm.models.eva import checkpoint_filter_fn
model = None
for name in ('vit_large_patch16_dinov3', 'vit_large_patch16_dinov3_qkvb'):
    for convert in (False, True):
        candidate = timm.create_model(name, pretrained=False, num_classes=0, img_size=336)
        sd = checkpoint_filter_fn(dict(state), candidate) if convert else dict(state)
        try:
            candidate.load_state_dict(sd, strict=True)
        except RuntimeError as e:
            print(f'  {name} convert={convert}: {str(e)[:260].replace(chr(10), " ")}')
            continue
        model, loaded_as = candidate, f'{name} convert={convert}'
        break
    if model is not None:
        break
assert model is not None, 'the checkpoint does not load strictly into any timm DINOv3 ViT-L variant'
print('LOADED strictly as', loaded_as, '| prefix tokens', model.num_prefix_tokens, '| features', model.num_features)

backbone = PooledBackbone(model).cuda()


In [ ]:
# The corpus, in parts so a late failure costs one part, not the run.
from torch.utils.data import DataLoader

ds = PreppedSlotDataset(uids, NPZ_ROOT, labels_df=None, n_groups=9, out_size=336,
                        slots=slot_names, augment=False)
loader = DataLoader(ds, batch_size=None, shuffle=False, num_workers=4, prefetch_factor=2)
PART = 1100
os.makedirs('/kaggle/working/features', exist_ok=True)
feats, masks, part_uids, parts = [], [], [], []
t0 = time.time()
for i, (image, mask, labels, uid) in enumerate(loader, 1):
    with torch.autocast('cuda', dtype=torch.float16):
        f = extract_study_features(backbone, image.cuda(non_blocking=True), mask.cuda())
    feats.append(f.float().cpu().numpy().astype(np.float16)); masks.append(mask.numpy()); part_uids.append(uid)
    if i % 200 == 0:
        el = time.time() - t0
        print(f'{i}/{len(uids)} studies, {el / 60:.1f} min, ETA {el / i * (len(uids) - i) / 60:.1f} min', flush=True)
    if len(feats) == PART or i == len(uids):
        arr = np.stack(feats)
        assert np.isfinite(arr.astype(np.float32)).all(), 'non-finite features'
        path = f'/kaggle/working/features/part{len(parts)}.npz'
        save_feature_cache(path, arr, np.stack(masks), part_uids, backbone='dinov3_vit_large_patch16@336')
        parts.append(path); feats, masks, part_uids = [], [], []
        print('wrote', path, flush=True)
print(f'extraction done in {(time.time() - t0) / 3600:.2f} h')


In [ ]:
# Merge the parts into the one cache every consumer reads, in the corpus's sorted uid order.
caches = [load_feature_cache(p) for p in parts]
features = np.concatenate([c.features for c in caches]); mask = np.concatenate([c.mask for c in caches])
merged_uids = [u for c in caches for u in c.uids]
assert merged_uids == uids, 'merged uid order differs from the sorted corpus order'
OUTF = '/kaggle/working/features/feat_dinov3l_336.npz'
save_feature_cache(OUTF, features, mask, merged_uids, backbone='dinov3_vit_large_patch16@336')
check = load_feature_cache(OUTF, expect_uids=uids)
print(f'cache {check.features.shape} {check.features.dtype}, {os.path.getsize(OUTF) / 2**30:.2f} GB; present slots/study {check.mask.sum(1).mean():.2f}')
for p in parts:
    os.remove(p)
print('CORPUS DONE | weights', loaded_as)
